# FL-PUB-005.1 — Reproducibility audit

*Epic EPIC-001 · story fl_pub_005 · all results below are loaded from files produced by the task's experiment function and labelled MEASURED/SIMULATED; clients are virtual; data is synthetic.*

## 1. Research objective
Verify determinism, re-execution, provenance consistency and artifact integrity of all stored results.

## 2. Scientific background
Reproducibility practices in ML [pineau2020, henderson2018].

References use keys of `docs/references/references_master.md` (status per entry).

## 3. Research questions
- **RQ1.** Are results bit-reproducible with a fixed seed?
- **RQ2.** Do stored summaries equal recomputation from raw runs?

## 4. Hypotheses
- **H1.** All determinism and consistency checks pass; the git-commit check fails (no repository).

## 5. Mathematical formulation
Equality checks: max|w_a − w_b| = 0; SHA-256 of config JSON recomputed.

## 6. Experimental environment

In [1]:
import os, sys, json, platform, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
def _find_root():
    p = Path(os.environ.get("FL_ROOT") or os.environ.get("RS_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set RS_ROOT")
ROOT = _find_root()
sys.path.insert(0, str(ROOT / "src"))
STORY = "fl_pub_005"
_cands = [ROOT / "experiments" / "results" / STORY]
RES = next(c for c in _cands if c.exists())
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
SEED = 0; np.random.seed(SEED)
def show_csv(name, n=12):
    p = RES / name
    if p.exists(): display(pd.read_csv(p).round(4).head(n))
    else: print("MISSING", name)
def show_fig(name):
    for base in (RES / "figures", RES):
        if (base / name).exists(): display(Image(filename=str(base / name))); return
    print("MISSING figure", name)
print("root:", ROOT, "| results:", RES)

root: /home/claude/fl_sprint | results: /home/claude/fl_sprint/experiments/results/fl_pub_005


In [2]:
import importlib.metadata as md
vers = {}
for p in ["numpy","scipy","scikit-learn","torch","tenseal","phe","pandas","matplotlib","cryptography","psutil","mlflow","opacus"]:
    try: vers[p] = md.version(p)
    except Exception: vers[p] = "not installed"
print(json.dumps(dict(python=platform.python_version(), platform=platform.platform(), versions=vers, seed=SEED, gpu="none (CPU only; GPU metrics NOT_MEASURED)"), indent=1))

{
 "python": "3.13.18",
 "platform": "Linux-6.18.44-fc-v338-x86_64-with-glibc2.39",
 "versions": {
  "numpy": "2.5.3",
  "scipy": "1.18.1",
  "scikit-learn": "1.9.1",
  "torch": "2.15.0",
  "tenseal": "0.3.18",
  "phe": "1.5.0",
  "pandas": "3.0.5",
  "matplotlib": "3.11.2",
  "cryptography": "50.0.1",
  "psutil": "7.2.2",
  "mlflow": "3.18.0",
  "opacus": "1.6.0"
 },
 "seed": 0,
 "gpu": "none (CPU only; GPU metrics NOT_MEASURED)"
}


## 7. Dataset description
Synthetic binary tabular dataset **SYN-IDS-v1** (scikit-learn `make_classification`, 40 features, 70:30 split). Chosen because no real IDS/clinical dataset was available in the sandbox. It has no clinical meaning; conclusions about absolute accuracy do not transfer.

In [3]:
from federated_learning.synthetic_data.generators.syn_ids_v1 import make_dataset, DATASET_ID
from federated_learning.data_partitioning.iid import partition_iid
from federated_learning.data_partitioning.dirichlet import partition_dirichlet
from federated_learning.data_partitioning.metrics import label_skew_stat
data = make_dataset(n_samples=6000, seed=SEED)
print(DATASET_ID, "train", data.X_train.shape, "test", data.X_test.shape, "positive rate", float(data.y_train.mean()))
print("NOTE: SYNTHETIC tabular data (not CIC-IDS-2017, not patient data); clients are VIRTUAL, not hospitals.")

SYN-IDS-v1 train (4200, 40) test (1800, 40) positive rate 0.40095238095238095
NOTE: SYNTHETIC tabular data (not CIC-IDS-2017, not patient data); clients are VIRTUAL, not hospitals.


## 8. Configuration and reproducibility settings
Seeds are fixed; each stored result carries `provenance` (experiment id, SHA-256 of the configuration, environment versions; `git_commit` is `null` because the workspace is not a git repository).

In [4]:
print("No FL configuration is needed to read stored results; see the stored JSON `provenance` blocks.")
print(json.dumps(json.load(open(next(RES.glob("*.json"))))["provenance"]["config"], indent=1)[:800])

No FL configuration is needed to read stored results; see the stored JSON `provenance` blocks.
{
 "K": 20,
 "rounds": 20,
 "seeds": 3,
 "sigma": 0.5,
 "target": 0.85,
 "byz_frac": 0.2,
 "backdoor": "medium"
}


## 9. Data preparation
Partitioning into virtual clients (IID or Dirichlet label skew) is performed by `federated_learning.data_partitioning`.

In [5]:
parts_iid = partition_iid(data.y_train, 10, SEED)
parts_dir = partition_dirichlet(data.y_train, 10, 0.3, SEED)
print('label-skew statistic IID:', round(label_skew_stat(data.y_train, parts_iid),3), ' Dirichlet(0.3):', round(label_skew_stat(data.y_train, parts_dir),3))

label-skew statistic IID: 0.019  Dirichlet(0.3): 0.363


## 10. Implementation
Reusable code lives in `src/federated_learning/` (`experiments/common.py`, `fl.py`). The cell below exercises it live (small, fast, deterministic) and asserts correctness.

In [6]:
from federated_learning.training.config import FLConfig
from federated_learning.servers.orchestrator import run_fl
parts = partition_iid(data.y_train, 5, SEED)
a = run_fl(FLConfig(rounds=2, seed=7), data, parts); b = run_fl(FLConfig(rounds=2, seed=7), data, parts); c = run_fl(FLConfig(rounds=2, seed=8), data, parts)
print("same seed identical:", bool(np.array_equal(a.w, b.w)), "| different seed differs:", bool(not np.array_equal(a.w, c.w))); assert np.array_equal(a.w, b.w)

same seed identical: True | different seed differs: True


## 11. Experiment execution
The full experiment was executed by `python -m federated_learning.pipelines.experiments.run_all --task FL-PUB-005.1` (see `logs/`). Its raw per-run results are loaded here; they are not re-computed inside the notebook.

In [7]:
show_csv('task_5_1_artifact_hashes.csv')

,path,sha256,bytes
0,outputs/research_sprint/fl_attack_003/figures/...,7e74f16bea057fc81120fdbba9d2d685d6ebd99f12ea02...,148173
1,outputs/research_sprint/fl_attack_003/figures/...,862fa5895dac0f6125d03c22de47825ff98fcf249902ee...,134894
2,outputs/research_sprint/fl_attack_003/figures/...,514d38e5f3ba18e3f4d21d26ad690e1e4263a960301e81...,171970
3,outputs/research_sprint/fl_attack_003/figures/...,dfb6253d2e732cd02d67ab23110763f762208b452e0f09...,171255
4,outputs/research_sprint/fl_attack_003/figures/...,548806028c655d89ada7b38a2fe8fc2497896257605a8b...,120408
5,outputs/research_sprint/fl_attack_003/task_3_1...,066f8371f9515e5358f8b629910f7fbd80b164a7bd19e7...,5930
6,outputs/research_sprint/fl_attack_003/task_3_1...,cc3caed54a9f634fb7db17c1c12247175b4648ee04b0ab...,603
7,outputs/research_sprint/fl_attack_003/task_3_1...,b7554c2f9c7f363579820c7ae4ae8aad7c048afee6e3e2...,966
8,outputs/research_sprint/fl_attack_003/task_3_1...,2cc41c6baa58095a79b53b866f31949123363837726331...,405
9,outputs/research_sprint/fl_attack_003/task_3_1...,fe411ec22bb512931d189675246806f886e228ad08f18a...,226


## 12. Metrics collection
Metric definitions: pass/fail per check with detail.

In [8]:
J = json.load(open(RES / 'task_5_1_reproducibility.json')); display(pd.DataFrame(J['checks']))

,check,passed,detail
0,same seed => bit-identical global model,True,max|diff|=0.0
1,different seed => different model,True,
2,dataset generation deterministic,True,
3,stored FedProx run re-executes to the stored a...,True,stored=0.885278 re-run=0.885278
4,provenance present and config hash recomputes,True,27 result files; inconsistent=[]
5,no result file unreadable,True,[]
6,git commit identifier available,False,workspace is not a git repository => commit re...
7,fl_sec_001: summary mean == mean of raw runs (...,True,gap=0.00e+00
8,fl_bench_004: summary mean == mean of raw runs...,True,gap=2.22e-16
9,notebooks present,False,"0 notebooks, 0 executed copies"


## 13. Statistical analysis
Not applicable (deterministic checks).

In [9]:
print('n/a')

n/a


## 14. Visualizations

In [10]:
print('no figure for this task (tabular result)')

no figure for this task (tabular result)


## 15. Interpretation of results
The numbers are computed in the next cell from the stored data; the text interprets only what that cell prints.

Passed checks and the failed git-commit check are printed.

In [11]:
J = json.load(open(RES / 'task_5_1_reproducibility.json')); ch = J['checks']; print(sum(c['passed'] for c in ch), 'of', len(ch), 'checks passed'); print([c['check'] for c in ch if not c['passed']])

9 of 11 checks passed
['git commit identifier available', 'notebooks present']


## 16. Limitations
- Bit-reproducibility holds on this software stack/CPU only; other BLAS/torch builds may differ.
- No git commit id.
- Synthetic data, virtual clients on one machine, 2 CPU cores, no GPU; no real network. Comm. bytes are logical payload sizes.
- No regulatory/compliance conclusion can be drawn from this notebook.

## 17. Conclusions
Results are reproducible on this stack; failures are listed honestly.

**Jira acceptance:** see `docs/fl_pub_005/results_and_discussion.md` for the criterion-by-criterion verdict (MET / PARTIALLY MET / NOT MET).

## 18. References
`pineau2020`, `henderson2018` — details in `docs/fl_pub_005/references.md`.